## Case Study Application – Axiom Insider Trading case

Testing pipeline against a known case of insider trading fraud on polymarket. 

Market question : "Will Axiom be accused of insider trading?"

In [ ]:
import requests, time, Path
import pandas as pd

In [ ]:
MARKET = "0x804bc34275d7d610dea3293c06105e925ebfcd6e454813aa27d59a882d50e86d"
YES_TOKEN = "103085646832854751469717081566514513444624814137586278494175134470235906780198"
TRADES_URL = "https://data-api.polymarket.com/v2/trades"
CSV_PATH = CSV_PATH = Path("analysis/data/axiom_trades.csv")

### Data Backfill

In [6]:
def fetch_all_trades(market: str, max_pages: int = 500) -> list[dict]:
    cursor = None 
    all_rows = []

    for page in range(max_pages): 
        params = {"condition": market, "limit": 1000} 
        if cursor is not None: 
            params['cursor'] = cursor

        r = requests.get(TRADES_URL, params=params, timeout=30)

        # If hit rate limit, wait & retry 
        if r.status_code == 429:              
            wait = float(r.headers.get("Retry-After", 5)) # Fallback to 5s if header is missing
            time.sleep(wait)
            continue  

        # Fetch data
        r.raise_for_status()
        body = r.json()
        rows = body['data']
        all_rows.extend(rows)
        print(f"page {page}: +{len(rows)} rows (total {len(all_rows)})")

        # Move to next page (if there is still)
        cursor = body["pagination"]["next_cursor"]
        if cursor is None: 
            break

        time.sleep(0.2)  

    return all_rows     

In [7]:
if CSV_PATH.exists(): 
    df = pd.read_csv(CSV_PATH, dtype={"token_id": str})
else: 
    rows = fetch_all_trades(market=MARKET)
    df = pd.DataFrame(rows)
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CSV_PATH, index=False)

print(len(df), "trades in total")
print((df["token_id"] == YES_TOKEN).sum(), "trades on the YES token")

AttributeError: 'str' object has no attribute 'exists'